# 02. Modélisation et Tracking MLflow

Ce notebook présente la phase d'expérimentation, d'optimisation et d'évaluation des modèles pour la prédiction du chiffre d'affaires des restaurants.

## Objectifs de ce notebook :
1. Charger les données prétraitées depuis `data/processed/train_processed.csv`.
2. Définir une stratégie de validation croisée rigoureuse (5-Fold).
3. Entraîner et comparer plusieurs algorithmes : Ridge, Lasso et Random Forest (les versions XGBoost et LightGBM requièrent OpenMP).
4. Enregistrer systématiquement les hyperparamètres et les métriques de performance (RMSE, MAE, R²) dans **MLflow**.
5. Identifier le meilleur modèle et le sauvegarder.

## 1. Imports et Configuration de MLflow

In [1]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.pipeline import Pipeline
import joblib

# Ajouter src au path pour charger le package local
sys.path.append(os.path.abspath("../src"))

# Import du preprocessing du package
from restaurant_revenue.features.preprocessing import get_preprocessor_pipeline

# Configuration de MLflow
mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("Restaurant_Revenue_Prediction")

/Users/mouhamadoulaminendiaye/workspace/dic3-tps/mlops/final_project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<Experiment: artifact_location='/Users/mouhamadoulaminendiaye/workspace/dic3-tps/mlops/final_project/mlruns/1', creation_time=1779909202205, experiment_id='1', last_update_time=1779909202205, lifecycle_stage='active', name='Restaurant_Revenue_Prediction', tags={}, trace_location=None, workspace='default'>

## 2. Chargement des données

In [2]:
from restaurant_revenue.features.preprocessing import process_and_save_data

processed_train_path = Path("../data/processed/train_processed.csv")
processed_test_path = Path("../data/processed/test_processed.csv")

# Exécution du pipeline du package à la volée si nécessaire
if not processed_train_path.exists() or not processed_test_path.exists():
    print("Données prétraitées introuvables. Lancement du pipeline global de traitement...")
    process_and_save_data()

df = pd.read_csv(processed_train_path)
X = df.drop(columns=["revenue"])
y = df["revenue"]

print(f"Dimensions de X : {X.shape}")
print(f"Dimensions de y : {y.shape}")
df.head()

Données prétraitées introuvables. Lancement du pipeline global de traitement...
Processed train and test datasets saved to /Users/mouhamadoulaminendiaye/workspace/dic3-tps/mlops/final_project/data/processed
Dimensions de X : (137, 40)
Dimensions de y : (137,)


,City Group,Type,P1,P2,P3,P4,P5,P6,P7,P8,...,P30,P31,P32,P33,P34,P35,P36,P37,revenue,days_since_open
0,Big Cities,IL,4,5.0,4.0,4.0,2,2,5,4,...,5,3,4,5,5,4,3,4,5653753.0,5647
1,Big Cities,FC,4,5.0,4.0,4.0,1,2,5,5,...,0,0,0,0,0,0,0,0,6923131.0,2513
2,Other,IL,2,4.0,2.0,5.0,2,3,5,5,...,0,0,0,0,0,0,0,0,2055379.0,663
3,Other,IL,6,4.5,6.0,6.0,4,4,10,8,...,25,12,10,6,18,12,12,6,2675511.0,1064
4,Other,IL,3,4.0,3.0,4.0,2,2,5,5,...,5,1,3,2,3,4,3,3,4316715.0,2063


## 3. Définition de l'entraînement et évaluation

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

models_config = {
    "Ridge": {
        "model": Ridge(alpha=1.0),
        "scale_numeric": True,
        "add_flags": False
    },
    "Lasso": {
        "model": Lasso(alpha=0.1, max_iter=10000),
        "scale_numeric": True,
        "add_flags": True
    },
    "RandomForest": {
        "model": RandomForestRegressor(n_estimators=100, max_depth=6, random_state=42),
        "scale_numeric": False,
        "add_flags": False
    }
}

# Ajout conditionnel des modèles nécessitant libomp
try:
    from xgboost import XGBRegressor
    models_config["XGBoost"] = {
        "model": XGBRegressor(n_estimators=100, max_depth=4, learning_rate=0.05, random_state=42),
        "scale_numeric": False,
        "add_flags": False
    }
    print("XGBoost chargé avec succès.")
except Exception:
    print("XGBoost indisponible (OpenMP manquant), ignoré dans les tests.")

try:
    from lightgbm import LGBMRegressor
    models_config["LightGBM"] = {
        "model": LGBMRegressor(n_estimators=100, max_depth=4, learning_rate=0.05, random_state=42, verbose=-1),
        "scale_numeric": False,
        "add_flags": False
    }
    print("LightGBM chargé avec succès.")
except Exception:
    print("LightGBM indisponible (OpenMP manquant), ignoré dans les tests.")

## 4. Exécution de la validation croisée et tracking MLflow

In [ ]:
results = {}

for name, config in models_config.items():
    print(f"Entraînement de {name}...")
    with mlflow.start_run(run_name=name):
        # Log des paramètres généraux
        mlflow.log_param("model_name", name)
        mlflow.log_param("scale_numeric", config["scale_numeric"])
        mlflow.log_param("add_sparse_flags", config["add_flags"])
        
        # Log des hyperparamètres du modèle
        for param, val in config["model"].get_params().items():
            if isinstance(val, (int, float, str, bool)) or val is None:
                mlflow.log_param(f"param_{param}", val)
                
        oof_preds = np.zeros(len(X))
        
        for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
            X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
            
            # Entraînement sur le log de la variable cible
            y_train_log = np.log1p(y_train)
            
            preprocessor = get_preprocessor_pipeline(
                add_sparse_flags=config["add_flags"],
                scale_numeric=config["scale_numeric"]
            )
            
            pipeline = Pipeline([
                ("preprocessor", preprocessor),
                ("regressor", config["model"])
            ])
            
            pipeline.fit(X_train, y_train_log)
            
            # Prédiction et reconversion exponentielle
            preds_log = pipeline.predict(X_val)
            oof_preds[val_idx] = np.expm1(preds_log)
            
        # Calcul des métriques globales
        rmse = np.sqrt(mean_squared_error(y, oof_preds))
        mae = mean_absolute_error(y, oof_preds)
        r2 = r2_score(y, oof_preds)
        
        results[name] = {"RMSE": rmse, "MAE": mae, "R2": r2, "predictions": oof_preds}
        print(f"{name} -> RMSE: {rmse:,.2f} | MAE: {mae:,.2f} | R2: {r2:.4f}")
        
        # Log des métriques dans MLflow
        mlflow.log_metric("oof_rmse", rmse)
        mlflow.log_metric("oof_mae", mae)
        mlflow.log_metric("oof_r2", r2)
        
        # Entraînement sur la totalité du dataset
        full_preprocessor = get_preprocessor_pipeline(
            add_sparse_flags=config["add_flags"],
            scale_numeric=config["scale_numeric"]
        )
        full_pipeline = Pipeline([
            ("preprocessor", full_preprocessor),
            ("regressor", config["model"])
        ])
        full_pipeline.fit(X, np.log1p(y))
        
        # Sauvegarde du modèle dans MLflow
        mlflow.sklearn.log_model(full_pipeline, artifact_path="model")

## 5. Visualisation des résultats

In [ ]:
metrics_df = pd.DataFrame({
    name: {"RMSE": res["RMSE"], "MAE": res["MAE"], "R2": res["R2"]}
    for name, res in results.items()
}).T

print("Synthèse des performances :")
display(metrics_df)

# Graphique de comparaison des RMSE
plt.figure(figsize=(8, 4))
sns.barplot(x=metrics_df.index, y=metrics_df["RMSE"], palette="viridis")
plt.title("Comparaison de la RMSE des modèles (OOF)")
plt.ylabel("RMSE (Revenu brut)")
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda x, loc: f"{x:,.0f}"))
plt.tight_layout()
plt.show()

## 6. Analyse des prédictions du meilleur modèle

In [ ]:
best_model_name = metrics_df["RMSE"].idxmin()
best_preds = results[best_model_name]["predictions"]

plt.figure(figsize=(8, 6))
plt.scatter(y, best_preds, alpha=0.7, color='steelblue', edgecolors='white', s=50)
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', lw=2)
plt.xlabel('Revenu Réel')
plt.ylabel('Revenu Prédit')
plt.title(f'Prédictions vs Réalité — {best_model_name}')
plt.gca().xaxis.set_major_formatter(plt.FuncFormatter(lambda x, loc: f"{x:,.0f}"))
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda x, loc: f"{x:,.0f}"))
plt.tight_layout()
plt.show()